# mp-spawn-workers — worked example 1: Use mp.spawn to collect per-rank results through a shared dict

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mp-spawn-workers`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`torch.multiprocessing.spawn(fn, args=(...,), nprocs=N, join=True)` launches N child processes, each running `fn(rank, *args)`. Because child processes have separate memory, you need an explicit IPC mechanism to pass results back to the parent. A `multiprocessing.Manager().dict()` (a managed shared dict) is the simplest Colab-safe way to collect one value per rank.

## Worked solution

**Step 1 — write the worker function to a module file.** `mp.spawn` pickles the function reference and sends it to child processes, which then re-import the module to unpickle. Functions defined in a notebook cell live in `__main__` and are not importable by a fresh child process. Writing to a temp `.py` file and importing it solves this.

**Step 2 — create the shared Manager dict.** `multiprocessing.Manager().dict()` creates a server-backed dict that all processes can read and write. We pass it as an argument through `mp.spawn`'s `args` tuple.

**Step 3 — each worker writes its rank to the shared dict.** Inside the worker: `shared_dict[rank] = rank * rank`. This runs independently in each process.

**Step 4 — call `mp.spawn(..., join=True)`.** With `join=True`, the parent blocks until all workers finish, so the shared dict is fully populated by the time the next line runs.

**Step 5 — read results in the parent.** After `spawn` returns, `shared_dict` contains one entry per rank.

In [ ]:
import sys
import importlib
import multiprocessing
import torch.multiprocessing as mp

WORKER_SRC = '''
import torch.distributed as dist
import os

def worker(rank, world_size, port, shared_dict):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group('gloo', rank=rank, world_size=world_size)
    # Each worker writes rank^2 to the shared dict
    shared_dict[rank] = rank * rank
    dist.destroy_process_group()
'''

def launch_and_collect(port: int = 29501) -> dict:
    # Write worker to importable module file
    with open('/tmp/dd_collect_worker.py', 'w') as f:
        f.write(WORKER_SRC)
    if '/tmp' not in sys.path:
        sys.path.insert(0, '/tmp')
    if 'dd_collect_worker' in sys.modules:
        mod = importlib.reload(sys.modules['dd_collect_worker'])
    else:
        mod = importlib.import_module('dd_collect_worker')

    manager = multiprocessing.Manager()
    shared_dict = manager.dict()

    mp.spawn(mod.worker, args=(2, port, shared_dict), nprocs=2, join=True)
    return dict(shared_dict)

try:
    results = launch_and_collect(port=29501)
    print("Results per rank:", results)
    # rank 0: 0^2=0, rank 1: 1^2=1
except Exception as e:
    print(f"Spawn not available in this environment: {e}")
    print("(Expected in restricted CPU-only Colab sessions.)")
    results = {0: 0, 1: 1}  # show expected output
    print("Expected results:", results)